# Deutsche Bibliotheksstatistik

In [179]:
import csv
import io
import requests
from itertools import islice
from collections import defaultdict
from matplotlib import pyplot as plt

In [145]:
years = list(range(1999, 2026))
years_csv = ",".join(map(str, years))

sql = (
    "me.fb = 'WB' and iso in ('DEU') and me.inr != 'DBI' and ("
    + " or ".join(
        f"fb_WB_{year}.inr is not null" for year in years
    )
    + ") and typ.nummer in (60)"
)

data = {
    "fragebogen": "WB",
    "jahre": years_csv,
    "attribute": "f09_bestand_buecher,f10_zu,f11_ab",
    "sql": sql,
    "bibs": "'AG188'",
}

In [146]:
response = requests.post(
    "https://www.bibliotheksstatistik.de/vaCsv",
    data=data
)

In [186]:
reader = csv.reader(io.StringIO(response.text), delimiter=";")
data = defaultdict(lambda: defaultdict(int))

titel, jahr, wert = list(islice(reader,2,5))
for t,j,w in zip(titel,jahr,wert):
    try:
        data[t][int(j)] = int(w.replace('.',''))
    except ValueError:
        pass

In [ ]:
for k,v in data.items():
    if len(v) > 0:
        series = [ v.get(y,0) for y in years ]
        plt.bar(years, series, label = k)
plt.title("Entwicklung des Printbestands an der UB der FU laut DBS")
plt.legend()
plt.show()